In [ ]:
from dlfs.layers import DenseLayer
from dlfs.activation import ReLU
from dlfs.loss import MSE_Loss
from dlfs.optimizers import Optimizer_SGD, Optimizer_Adam
from dlfs.model import SequentialModel
from sklearn.preprocessing import StandardScaler

from viz_helpers import *
from dataset_helpers import *

# Dense Layer visualisation for regression

- this notebook aims to visualize 1D, 2D, 3D linear transformations (by Dense Layers) and activation functions applied to regression data 

# Different datasets for visualisation

In [ ]:
datasets = {
    "linear": make_linear_regression,
    "sine": make_sine,
    "cubic": make_cubic,
    "logarithm": make_logarithm
}

# Regression dataset visualisation

In [ ]:
dataset = "sine"
n_samples = 150
n_features = 2
noise = 0.3
resolution = 50

X, y, true_f = datasets[dataset](n_samples, n_features, noise)
title = f"{str.capitalize(dataset)} data"

if n_features == 1:
    X = X.reshape(-1, 1)
    plot_2d_reg_problem(X, y, f=true_f, title=title, curve_label="True f", resolution=resolution)
elif n_features == 2:
    plot_3d_reg_problem(X, y, f=true_f, title=title, surface_label="True f", resolution=resolution)

# Normalize target variable

In [ ]:
scaler = StandardScaler()
y_scaled = scaler.fit_transform(y.reshape(-1, 1))

# Training regression model

In [ ]:
layers = [
    DenseLayer(n_features, 3), 
    ReLU(),
    DenseLayer(3, 2),
    ReLU(),
    DenseLayer(2, 1)
]

model = SequentialModel(
    layers=layers, 
    loss_function=MSE_Loss(), 
    optimizer=Optimizer_SGD(learning_rate=1e-3, momentum=0.3, decay=0.)
)

model.train(X, y_scaled, print_every=500, epochs=1000)

# Extract each layer's output

In [ ]:
model.forward(X)

Z1 = model.wrapper.layers[0].output.copy() # first dense layer
A1 = model.wrapper.layers[1].output.copy() # first dense + actiavtion
Z2 = model.wrapper.layers[2].output.copy() # second dense layer
A2 = model.wrapper.layers[3].output.copy() # second dense layer + activation
Z3 = model.wrapper.layers[4].output.copy() # third dense layer, final output

input_dim = n_features
layer1_dim = Z1.shape[1]
layer2_dim = Z2.shape[1]
layer3_dim = Z3.shape[1]

# Plotting input data

In [ ]:
title = f"{input_dim}D Input Data"

if input_dim == 1:
    plot_1d_regression_data(X, y, title)
elif input_dim == 2:
    plot_2d_regression_data(X, y, title, axis1="x1", axis2="x2")

# Plotting first linear transformation

In [ ]:
title = f"{input_dim}D → {layer1_dim}D using first Dense Layer"

if layer1_dim == 2:
    plot_2d_regression_data(Z1, y.reshape(-1), title, axis1="Neuron 1", axis2="Neuron 2")
elif layer1_dim == 3:
    plot_3d_regression_data(Z1, y.reshape(-1), title)

# Plotting first linear transformation and first activation

In [ ]:
title = f"{input_dim}D → {layer1_dim}D using first Dense Layer + Activation"

if layer1_dim == 2:
    plot_2d_regression_data(A1, y.reshape(-1), title, axis1="Neuron 1", axis2="Neuron 2")
elif layer1_dim == 3:
    plot_3d_regression_data(A1, y.reshape(-1), title)

# Plotting second linear transformation

In [ ]:
title = f"{layer1_dim}D → {layer2_dim}D using second Dense Layer"

if layer2_dim == 2:
    plot_2d_regression_data(Z2, y.reshape(-1), title, axis1="Neuron 1", axis2="Neuron 2")
elif layer2_dim == 3:
    plot_3d_regression_data(Z2, y.reshape(-1), title)

# Plotting second linear transformation and second activation

In [ ]:
title = f"{layer1_dim}D → {layer2_dim}D using second Dense Layer + Activation"

if layer2_dim == 2:
    plot_2d_regression_data(A2, y.reshape(-1), title, axis1="Neuron 1", axis2="Neuron 2")
elif layer2_dim == 3:
    plot_3d_regression_data(A2, y.reshape(-1), title)

# Plotting third linear transformation (network output)

In [ ]:
title = f"{layer2_dim}D → {layer3_dim}D using third Dense Layer"
plot_1d_regression_data(Z3, y.reshape(-1), title)

# Plotting third linear transformation (network output unscaled)

In [ ]:
title = f"{layer2_dim}D → {layer3_dim}D using third Dense Layer (Unscaled)"
Z3_unscaled = scaler.inverse_transform(Z3)
plot_1d_regression_data(Z3_unscaled, y.reshape(-1), title)

# Preprocess input data for final plot

In [ ]:
if n_features == 1:
    X_range = np.linspace(X.min(), X.max(), resolution).reshape(-1, 1)
    y_pred_scaled = model.predict(X_range)
    y_pred = scaler.inverse_transform(y_pred_scaled)
    network_curve = y_pred
    
elif n_features == 2:
    X1, X2 = create_meshgrid(X, resolution)
    X_range = np.stack((X1.ravel(), X2.ravel()), axis=-1)
    y_pred_scaled = model.predict(X_range)
    y_pred = scaler.inverse_transform(y_pred_scaled)
    network_surface = y_pred.reshape(X1.shape) 

# Visualize network output

In [ ]:
label = "Network prediction"

if n_features == 1:
    plot_2d_reg_problem(X, y, curve=network_curve, title="Original X and y", curve_label=label, resolution=resolution)
elif n_features == 2:
    plot_3d_reg_problem(X, y, surface=network_surface, title="Test", surface_label=label, resolution=resolution)